[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesShang/TorchCode/blob/master/solutions/67_fused_rmsnorm_residual_solution.ipynb)

# 🟡 Solution: Fused RMSNorm + Residual

Reference solution for `fused_rmsnorm_residual`.

In [ ]:
# Install the latest torch-judge from this repo in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q --force-reinstall --no-deps git+https://github.com/CharlesShang/TorchCode.git@master')
except ImportError:
    pass


In [ ]:
import torch


In [ ]:
# ✅ SOLUTION

def fused_rmsnorm_residual(x: torch.Tensor, residual: torch.Tensor, weight: torch.Tensor,
                           eps: float = 1e-6, residual_in_fp32: bool = True):
    if residual_in_fp32:
        updated = x.float() + residual.float()
        norm_input = updated
    else:
        updated = x + residual
        norm_input = updated
    rms = torch.rsqrt(norm_input.pow(2).mean(dim=-1, keepdim=True) + eps)
    y = norm_input * rms * weight.to(norm_input.dtype)
    return y.to(x.dtype), updated


In [ ]:
# Verify
x = torch.randn(2, 3, 8)
r = torch.randn(2, 3, 8)
w = torch.ones(8)
y, updated = fused_rmsnorm_residual(x, r, w)
print(y.shape, updated.shape)


In [ ]:
# Run judge
from torch_judge import check
check('fused_rmsnorm_residual')
